# Importo librerie necessarie


In [8]:
from re import match

from pymongo import MongoClient
from scripts import methods
from bson.objectid import ObjectId
from pandasql import sqldf
import pandas as pd

# Attivazione del Client MongoDB

In [9]:
# Chiamo il client
client = MongoClient("mongodb://localhost:27017/")

#Acquisisco il DB
db = client["Keyblade"]

#Prendo in esempio il dataset videogames_2016
videogames_2016 = db["videogames_2016"]
videogames_2024 = db["videogames_2024"]

# Query per la visualizzazione di giochi per rating

In [3]:
#Visualizzo i rating distinti presenti nel dataset videogames_2016
ratings=videogames_2016.distinct("Rating")

#creo un dizionario per la lista dei nomi per ciascun rating
rating_list={}

# Per ogni rating eseguo aggregazione
for rating in ratings:
    pipeline = [
        {"$match": {"Rating": rating}},
        {"$project": {"Name": 1, "Platform": 1, "_id": 0}}
    ]
    games = videogames_2016.aggregate(pipeline)
    games_df = pd.DataFrame(list(games))
    rating_list[rating] = games_df




# Visualizzo i giochi per rating

In [15]:
# Visualizzazione dei risultati
for rating, games_df in rating_list.items():
    print(f"Giochi con rating {rating}:")
    display(games_df)
    print("\n")

Giochi con rating AO:


,Name,Platform
0,Grand Theft Auto: San Andreas,XB




Giochi con rating E:


,Name,Platform
0,Wii Sports,Wii
1,Mario Kart Wii,Wii
2,Wii Sports Resort,Wii
3,New Super Mario Bros.,DS
4,Wii Play,Wii
...,...,...
3986,E.T. The Extra-Terrestrial,GBA
3987,Planet Monsters,GBA
3988,Bust-A-Move 3000,GC
3989,Mega Brain Boost,DS




Giochi con rating E10+:


...

,Name,Platform
0,Grand Theft Auto V,PS3
1,Grand Theft Auto: San Andreas,PS2
2,Grand Theft Auto V,X360
3,Grand Theft Auto: Vice City,PS2
4,Call of Duty: Modern Warfare 3,X360
...,...,...
1558,Xblaze: Lost Memories,PSV
1559,Metal Gear Solid HD Edition,X360
1560,Metal Gear Solid V: The Definitive Experience,XOne
1561,Mortal Kombat: Deadly Alliance,GBA




Giochi con rating RP:


,Name,Platform
0,Super Mario Bros.,NES
1,Pokemon Red/Pokemon Blue,GB
2,Tetris,GB
3,Duck Hunt,NES
4,Nintendogs,DS
...,...,...
6765,Samurai Warriors: Sanada Maru,PS3
6766,LMA Manager 2007,X360
6767,Haitaka no Psychedelica,PSV
6768,Spirits & Spells,GBA




Giochi con rating T:


,Name,Platform
0,Super Smash Bros. Brawl,Wii
1,Final Fantasy VII,PS
2,Gran Turismo 2,PS
3,Final Fantasy X,PS2
4,The Sims 3,PC
...,...,...
2956,Super Robot Taisen: Original Generation,GBA
2957,End of Nations,PC
2958,Outdoors Unleashed: Africa 3D,3DS
2959,Breach,PC


# Query per la visualizzazione delle vendite totali dei giochi in base al Developer (Video giochi 2016)

In [6]:
# Recupero dei developer distinti
developers = videogames_2016.distinct("Developer")

# Lista per raccogliere i risultati
rows = []

# Ciclo sui developer
for developer in developers:
    if developer == "Unknown":
        # Visualizza giochi singoli per Unknown
        pipeline = [
            {"$match": {"Developer": "Unknown"}},
            {"$project": {"Developer": 1,"Global_Sales": 1, "_id": 0}}
        ]
        result = list(videogames_2016.aggregate(pipeline))
        rows.extend(result)
    else:
        # Somma le vendite per gli altri developer
        pipeline = [
            {"$match": {"Developer": developer}},
            {"$group": {
                "_id": "$Developer",
                "Global_Sales": {"$sum": "$Global_Sales"}
            }}
        ]
        result = list(videogames_2016.aggregate(pipeline))
        if result:
            rows.append({"Developer": developer, "Global_Sales": result[0]["Global_Sales"]})
        else:
            rows.append({"Developer": developer,"Global_Sales": 0})

# Creazione del DataFrame
sales_df = pd.DataFrame(rows)




,Developer,Global_Sales
1020,Nintendo,531.71
449,EA Sports,175.38
431,EA Canada,142.32
1539,Ubisoft,132.54
1220,Rockstar North,119.47
...,...,...
316,Compulsion Games,0.01
267,"Camouflaj, LLC",0.01
444,EA Phenomic,0.01
364,"DMA Design, Rockstar North",0.01


# Visualizzazione delle vendite totali dei giochi in base al Publisher

In [ ]:
# Ordinamento per Global_Sales
sales_df = sales_df.sort_values(by="Global_Sales", ascending=False)

# Visualizzazione
display(sales_df)

# Query per la visualizzazione delle vendite totali dei giochi in base al Developer (Video giochi 2024)

In [11]:
# Recupero dei developer distinti
developers = videogames_2024.distinct("developer")

# Lista per raccogliere i risultati
rows = []

# Ciclo sui developer
for developer in developers:
    if developer == "Unknown":
        # Visualizza giochi singoli per Unknown
        pipeline = [
            {"$match": {"developer": "Unknown"}},
            {"$project": {"developer": 1, "total_sales": 1, "_id": 0}}
        ]
        result = list(videogames_2024.aggregate(pipeline))
        rows.extend(result)
    else:
        # Somma le vendite per gli altri developer
        pipeline = [
            {"$match": {"developer": developer}},
            {"$group": {
                "_id": "$developer",
                "total_sales": {"$sum": "$total_sales"}
            }}
        ]
        result = list(videogames_2024.aggregate(pipeline))
        if result:
            rows.append({"developer": developer, "total_sales": result[0]["total_sales"]})
        else:
            rows.append({"developer": developer, "total_sales": 0})

# Creazione del DataFrame
sales_df = pd.DataFrame(rows)



,developer,total_sales
2251,EA Canada,275.56
2269,EA Tiburon,178.33
7945,Ubisoft Montreal,172.96
7828,Treyarch,150.19
7816,Traveller's Tales,149.55
...,...,...
13281,team BitClub,0.00
13311,zerozerozero,0.00
1510,Click Entertainment,0.00
13309,yyr,0.00


# Visualizzazione delle vendite totali dei giochi in base al Publisher 2024

In [ ]:
# Ordinamento per total_sales
sales_df = sales_df.sort_values(by="total_sales", ascending=False)

# Visualizzazione
display(sales_df)


# Visualizzazione delle vendite dei giochi in base al Developer (Video giochi 2016)

In [5]:
# Pipeline: visualizza tutti i giochi con dettagli di vendita
pipeline = [
    {"$project": {
        "_id": 0,
        "Developer": 1,
        "Global_Sales": 1,
        "NA_Sales": 1,
        "EU_Sales": 1,
        "JP_Sales": 1,
        "Other_Sales": 1
    }},
    {"$sort": {"Global_Sales": -1}}
]#Inserendo il -1 nell'operazione di sort, i risultati vengono ordinati in ordine decrescente in base alle vendite globali

# Esecuzione della pipeline
result = list(videogames_2016.aggregate(pipeline))




,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Developer
0,41.36,28.96,3.77,8.45,82.53,Nintendo
1,29.08,3.58,6.81,0.77,40.24,Unknown
2,15.68,12.76,3.79,3.29,35.52,Nintendo
3,15.61,10.93,3.28,2.95,32.77,Nintendo
4,11.27,8.89,10.22,1.00,31.37,Unknown
...,...,...,...,...,...,...
16712,0.00,0.00,0.01,0.00,0.01,Unknown
16713,0.00,0.01,0.00,0.00,0.01,Unknown
16714,0.00,0.00,0.01,0.00,0.01,Unknown
16715,0.01,0.00,0.00,0.00,0.01,Unknown


# VIsualizzazione delle vendite dei giochi in base al Developer

In [ ]:
# Conversione in DataFrame
sales_df = pd.DataFrame(result)

# Visualizzazione
display(sales_df)